# NIST line list

Goal: test `line_list.get_lines`, which returns the absorption lines of a
species from the NIST Atomic Spectra Database.

The NIST tables are downloaded once per species by
`line_list.download_line_table` and saved in `data/line_lists/` (committed),
so `get_lines` works offline. Only electric-dipole transitions with both $f$
and $A_{ki}$ are kept. The damping constant $\Gamma$ is the sum of the decay
rates of the upper and lower levels, i.e. of $A_{ki}$ over every transition
NIST lists from each level. Wavelengths in Angstrom, vacuum.

In [1]:
import numpy as np
from astropy.table import Table

from spectral_diag.line_list import LineSelection, get_lines

## Lines of a few species

`f_min` drops the intercombination / very weak lines (e.g. SiII 2329, 2335).
The default `e_low_max = 0` keeps the lines from the ground level only
(resonance lines, the ones seen in cold ISM/CGM gas).

In [2]:
species = ["CIV", "SiII", "SII", "FeII"]
selection = LineSelection(wave_min=1150, wave_max=2700, f_min=1e-3)
rows = []
for sp in species:
    for line in get_lines(sp, selection):
        rows.append((line.name, line.wave_rest, line.f, line.gamma))
lines_table = Table(rows=rows, names=["name", "wave_rest", "f", "gamma"])
lines_table["wave_rest"].info.format = ".3f"
lines_table["f"].info.format = ".3e"
lines_table["gamma"].info.format = ".3e"
lines_table

name,wave_rest,f,gamma
str13,float64,float64,float64
CIV 1548.204,1548.204,1.900e-01,2.650e+08
CIV 1550.784,1550.784,9.520e-02,2.640e+08
SiII 1190.416,1190.416,2.770e-01,4.103e+09
SiII 1193.290,1193.290,5.750e-01,4.090e+09
SiII 1260.422,1260.422,1.220e+00,3.043e+09
SiII 1304.370,1304.370,9.280e-02,9.870e+08
SiII 1526.707,1526.707,1.330e-01,1.133e+09
SiII 1808.013,1808.013,2.490e-03,2.863e+06
SII 1250.584,1250.584,6.020e-03,5.130e+07


FeII oscillator strengths in NIST are more recent than those of
Verner+96, the table used before (up to ~30 % different for 1608, 2374,
2586). Since N scales as 1/f in the linear regime, the source matters.
FeII 2367 is missing: NIST gives no $f$ for it.

## Hydrogen: multiplet totals

NIST splits each hydrogen line into its fine-structure components
(H$\alpha$: 7 components within 0.2 Angstrom, ~10 km/s), far below the
resolution of SDSS (~150 km/s) or COS (~17 km/s). Their $f$ cannot simply
be added: each refers to its own lower sublevel. `multiplet_totals=True`
keeps instead the single $n \to n'$ line that NIST lists for hydrogenic
ions, with the total $f$ (H$\alpha$: 0.641).

In [3]:
sdss_range = {"wave_min": 3600, "wave_max": 9300, "e_low_max": np.inf}
components = get_lines("HI", LineSelection(**sdss_range))
totals = get_lines("HI", LineSelection(**sdss_range, multiplet_totals=True))
print(f"{len(components)} fine-structure components, {len(totals)} multiplet totals")

halpha_components = [line for line in components if abs(line.wave_rest - 6564.6) < 1]
halpha_total = [line for line in totals if abs(line.wave_rest - 6564.6) < 1]
print(f"Halpha: {len(halpha_components)} components, sum of f = "
      f"{sum(line.f for line in halpha_components):.3f} (wrong)")
print(f"Halpha total: f = {halpha_total[0].f:.4f}")

70 fine-structure components, 50 multiplet totals
Halpha: 7 components, sum of f = 1.855 (wrong)
Halpha total: f = 0.6411


### Check against the Rydberg formula

$1/\lambda = R_\mathrm{H}\,(1/n_\mathrm{low}^2 - 1/n_\mathrm{up}^2)$, with
$R_\mathrm{H} = R_\infty / (1 + m_e/m_p)$. The NIST totals differ from these
Bohr wavelengths by a few km/s only (fine structure and Lamb shift).

In [4]:
from scipy import constants as scipy_constants

from spectral_diag import constants

hydrogen_rydberg = scipy_constants.Rydberg * 1e-10 / (1 + scipy_constants.m_e / scipy_constants.m_p)
SERIES = {2: "Balmer", 3: "Paschen"}
MAX_UPPER_LEVEL = 60
FIRST_LINES_SHOWN = 4


def rydberg_wavelength(n_lower, n_upper):
    """Vacuum wavelength of the hydrogen line n_upper -> n_lower, Angstrom (Bohr model)."""
    return 1 / (hydrogen_rydberg * (1 / n_lower**2 - 1 / n_upper**2))


for n_lower, series in SERIES.items():
    print(f"{series}: first {FIRST_LINES_SHOWN} lines in the SDSS range")
    upper_levels_in_range = [
        n_upper
        for n_upper in range(n_lower + 1, MAX_UPPER_LEVEL + 1)
        if rydberg_wavelength(n_lower, n_upper) >= sdss_range["wave_min"]
        and rydberg_wavelength(n_lower, n_upper) <= sdss_range["wave_max"]
    ]
    for n_upper in upper_levels_in_range[:FIRST_LINES_SHOWN]:
        rydberg_wave = rydberg_wavelength(n_lower, n_upper)
        closest = min(totals, key=lambda line: abs(line.wave_rest - rydberg_wave))
        offset_kms = (closest.wave_rest / rydberg_wave - 1) * constants.C_KMS
        print(f"  {n_upper:2d} -> {n_lower}: Rydberg {rydberg_wave:9.3f} A, NIST {closest.wave_rest:9.3f} A, "
              f"offset {offset_kms:+.1f} km/s, f = {closest.f:.4f}")


Balmer: first 4 lines in the SDSS range
   3 -> 2: Rydberg  6564.696 A, NIST  6564.632 A, offset -2.9 km/s, f = 0.6411
   4 -> 2: Rydberg  4862.738 A, NIST  4862.691 A, offset -2.9 km/s, f = 0.1194
   5 -> 2: Rydberg  4341.730 A, NIST  4341.691 A, offset -2.7 km/s, f = 0.0447
   6 -> 2: Rydberg  4102.935 A, NIST  4102.899 A, offset -2.6 km/s, f = 0.0221
Paschen: first 4 lines in the SDSS range
   9 -> 3: Rydberg  9231.604 A, NIST  9231.546 A, offset -1.9 km/s, f = 0.0102
  10 -> 3: Rydberg  9017.440 A, NIST  9017.384 A, offset -1.9 km/s, f = 0.0070
  11 -> 3: Rydberg  8865.270 A, NIST  8865.216 A, offset -1.8 km/s, f = 0.0050
  12 -> 3: Rydberg  8752.928 A, NIST  8752.876 A, offset -1.8 km/s, f = 0.0037
